# Banco de Dados Nebulosos para identicação de risco de queimadas





primeiro foi pego dados do inep DB queimadas, porem na data escolhida não haviam atributos essenciais para pesquisa, sendo esses "diaSemchuva" e "preciptacao" que podem ser modelados logica fuzzy.
[fonte de dados](https://data.inpe.br/queimadas/bdqueimadas/#exportar-dados)

## Para fazer

#### Fase 1
- [x] carregar dados e limpar
- [x] conectar ao ArangoDB
- [ ] enviar dados pro arango

#### Fase 2 
- [ ] desenvolver modelo de fuzzificação.

----


## Carregar dados

In [1]:
import pandas as pd

df = pd.read_csv("data/bdqueimadas_2026-09-01_2026-10-26.csv")
df.head()


,DataHora,Satelite,Pais,Estado,Municipio,Bioma,DiaSemChuva,Precipitacao,RiscoFogo,FRP,Latitude,Longitude
0,2026/09/01 00:00:00,GOES-19,Brasil,MARANHÃO,COROATÁ,Cerrado,41.0,0.00,0.99,41.0,-3.8416,-44.0824
1,﻿2026/09/01 00:00:00,GOES-19,Brasil,MATO GROSSO,SANTO ANTÔNIO DO LESTE,Cerrado,50.0,0.90,1.00,85.0,-14.6925,-53.5384
2,2026/09/01 00:00:00,GOES-19,Brasil,GOIÁS,BALIZA,Cerrado,4.0,3.79,1.00,66.9,-16.1299,-52.3922
3,2026/09/01 00:00:00,GOES-19,Brasil,GOIÁS,BALIZA,Cerrado,4.0,3.85,1.00,75.5,-16.1303,-52.3707
4,2026/09/01 00:00:00,GOES-19,Brasil,GOIÁS,BALIZA,Cerrado,4.0,4.33,1.00,138.7,-16.1494,-52.3893


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1587617 entries, 0 to 1587616
Data columns (total 12 columns):
 #   Column        Non-Null Count    Dtype  
---  ------        --------------    -----  
 0   DataHora      1587617 non-null  str    
 1   Satelite      1587617 non-null  str    
 2   Pais          1587617 non-null  str    
 3   Estado        1587617 non-null  str    
 4   Municipio     1587617 non-null  str    
 5   Bioma         1587617 non-null  str    
 6   DiaSemChuva   1477614 non-null  float64
 7   Precipitacao  1477614 non-null  float64
 8   RiscoFogo     1477614 non-null  float64
 9   FRP           1536286 non-null  float64
 10  Latitude      1587617 non-null  float64
 11  Longitude     1587617 non-null  float64
dtypes: float64(6), str(6)
memory usage: 145.4 MB


In [3]:
import os
import numpy as np


DB_NAME = "queimadas"

NUMERIC_COLUMNS = [
    "DiaSemChuva",
    "Precipitacao",
    "RiscoFogo",
    "FRP"
]

df["Estado"] = df["Estado"].fillna("Desconhecido").astype(str).str.strip()

for col in NUMERIC_COLUMNS:
    df[col] = pd.to_numeric(df[col], errors="coerce")




# Conexão com Banco de dados

In [4]:
import os
from dotenv import load_dotenv
from arango import ArangoClient


load_dotenv()
db_password = os.getenv("DB_PASSWORD")
# Initialize the client for ArangoDB.
client = ArangoClient(hosts="http://localhost:8529")
user = 'root'


# Connect to "_system" database as root user.
sys_db = client.db("_system", username="root", password=db_password)


if not sys_db.has_database(DB_NAME):
    sys_db.create_database(DB_NAME)

db = client.db(DB_NAME, username=user, password=db_password)

# Enviando dados para o arango

In [11]:

import hashlib
import math
import numpy as np
import pandas as pd

# Criar coleções
if not db.has_collection("Municipios"):
    db.create_collection("Municipios")

if not db.has_collection("similaridades"):
    db.create_collection("similaridades", edge=True)


def limpar_valor(valor):
    """Converte valores para tipos compatíveis com JSON/ArangoDB."""

    # Valores ausentes do Pandas
    if valor is None or valor is pd.NA or valor is pd.NaT:
        return None

    # Converter arrays NumPy e listas recursivamente
    if isinstance(valor, np.ndarray):
        return [limpar_valor(v) for v in valor.tolist()]

    if isinstance(valor, (list, tuple)):
        return [limpar_valor(v) for v in valor]

    # Converter dicionários recursivamente
    if isinstance(valor, dict):
        return {str(k): limpar_valor(v) for k, v in valor.items()}

    # Converter números NumPy
    if isinstance(valor, np.integer):
        return int(valor)

    if isinstance(valor, np.floating):
        valor = float(valor)

    if isinstance(valor, np.bool_):
        return bool(valor)

    # Tratar NaN e infinito
    if isinstance(valor, float):
        return valor if math.isfinite(valor) else None

    # Tratar NaN/NaT escalares do Pandas
    if not isinstance(valor, (str, bytes, bool, int, float)):
        try:
            ausente = pd.isna(valor)
            if isinstance(ausente, (bool, np.bool_)) and ausente:
                return None
        except (TypeError, ValueError):
            pass

    return valor


# Agrupar eventos por município
municipios = {}

for (estado, nome), grupo in df.groupby(
    ["Estado", "Municipio"], sort=True
):
    raw_key = f"{estado}|{nome}"
    key = hashlib.sha1(raw_key.encode("utf-8")).hexdigest()

    eventos = []

    for _, row in grupo.iterrows():
        evento = {
            "dataHora": (
                str(row["DataHora"])
                if pd.notna(row["DataHora"]) else None
            ),
            "satelite": (
                str(row["Satelite"])
                if pd.notna(row["Satelite"]) else None
            ),
            "bioma": (
                str(row["Bioma"])
                if pd.notna(row["Bioma"]) else None
            ),
            "diaSemChuva": (
                float(row["DiaSemChuva"])
                if pd.notna(row["DiaSemChuva"]) else None
            ),
            "precipitacao": (
                float(row["Precipitacao"])
                if pd.notna(row["Precipitacao"]) else None
            ),
            "riscoFogo": (
                float(row["RiscoFogo"])
                if pd.notna(row["RiscoFogo"]) else None
            ),
            "frp": (
                float(row["FRP"])
                if pd.notna(row["FRP"]) else None
            ),
            "latitude": (
                float(row["Latitude"])
                if pd.notna(row["Latitude"]) else None
            ),
            "longitude": (
                float(row["Longitude"])
                if pd.notna(row["Longitude"]) else None
            )
        }

        eventos.append(evento)

    # Médias para o cálculo posterior de similaridade
    medias = grupo[NUMERIC_COLUMNS].mean()

    municipios[key] = {
        "_key": key,
        "nome": str(nome),
        "estado": str(estado),
        "quantidadeEventos": len(eventos),
        "eventos": eventos,
        "medias": medias.to_dict()
    }




colecao = db.collection("Municipios")

inseridos = 0
atualizados = 0
falhas = 0

for key, municipio in municipios.items():
    try:
        documento = limpar_valor(municipio)

        if colecao.has(key):
            colecao.update(documento)
            atualizados += 1
        else:
            colecao.insert(documento)
            inseridos += 1

    except Exception as e:
        falhas += 1
        print(f"\nErro no município: {municipio['nome']}")
        print(f"Estado: {municipio['estado']}")
        print(f"Erro: {e}")

print(f"Municípios preparados: {len(municipios)}")
print(f"Novos municípios inseridos: {inseridos}")
print(f"Municípios atualizados: {atualizados}")
print(f"Falhas: {falhas}")



Municípios preparados: 4395
Novos municípios inseridos: 0
Municípios atualizados: 4395
Falhas: 0
